# GloVe: Global Vectors for Word Representation

GloVe는 말뭉치 전체의 **단어-단어 동시 등장 통계(co-occurrence statistics)**를 이용해 단어를 고정 길이 벡터로 표현하는 비지도 학습 알고리즘이다.

이 노트북의 목표는 다음과 같다.

1. GloVe가 어떤 통계를 학습하는지 이해한다.
2. 작은 말뭉치에서 공기행렬을 직접 만든다.
3. PyTorch로 GloVe 목적함수를 구현해 벡터를 학습한다.
4. Gensim으로 공개 사전학습 GloVe 벡터를 사용한다.
5. 유사어, 단어 유추, 시각화, 문서 벡터화와 저장 방법을 익힌다.

공식 자료: [Stanford GloVe 프로젝트](https://nlp.stanford.edu/projects/glove/), [GloVe 논문](https://nlp.stanford.edu/pubs/glove.pdf)

## 1. GloVe와 Word2Vec의 차이

| 구분 | Word2Vec | GloVe |
|---|---|---|
| 핵심 아이디어 | 중심 단어와 주변 단어의 예측 문제 | 전체 말뭉치의 단어 공기 빈도를 행렬로 구성 |
| 주로 사용하는 정보 | 지역 문맥을 반복적으로 샘플링 | 집계된 전역 공기 통계 |
| 대표 목적함수 | Negative sampling 또는 hierarchical softmax | 가중 최소제곱 오차 |
| 결과 | 단어별 고정 길이 벡터 | 단어별 고정 길이 벡터 |
| 미등록 단어 | 기본적으로 벡터 없음 | 기본적으로 벡터 없음 |

두 방법 모두 학습이 끝나면 단어마다 하나의 정적(static) 벡터를 만든다. 따라서 문맥에 따라 의미가 달라지는 다의어 표현에는 한계가 있다. 예를 들어 `bank`는 금융기관과 강둑이라는 서로 다른 의미가 있어도 하나의 벡터를 사용한다.

## 2. GloVe의 수학적 원리

말뭉치의 vocabulary 크기를 $V$라고 하고, $X_{ij}$를 중심 단어 $i$의 주변에서 문맥 단어 $j$가 등장한 가중 횟수라고 하자. GloVe는 다음 목적함수를 최소화한다.

$$
J = \sum_{i=1}^{V}\sum_{j=1}^{V}
f(X_{ij})
\left(
w_i^{\mathsf T}\tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij}
\right)^2
$$

- $w_i$: 중심 단어 임베딩
- $\tilde{w}_j$: 문맥 단어 임베딩
- $b_i$, $\tilde{b}_j$: 중심·문맥 편향
- $X_{ij}$: 두 단어의 공기 빈도
- $f(X_{ij})$: 너무 드물거나 지나치게 흔한 단어 쌍의 영향을 조절하는 가중 함수

대표적인 가중 함수는 다음과 같다.

$$
f(x) =
\begin{cases}
\left(\dfrac{x}{x_{\max}}\right)^\alpha, & x < x_{\max} \\
1, & x \geq x_{\max}
\end{cases}
$$

논문에서는 보통 $\alpha=0.75$와 $x_{\max}=100$을 사용한다. 학습 후에는 중심 벡터와 문맥 벡터를 더한 $w_i + \tilde{w}_i$를 최종 단어 표현으로 많이 사용한다.

핵심은 내적과 편향의 합이 두 단어가 함께 등장한 횟수의 로그값에 가까워지도록 학습한다는 것이다. 공기 빈도 비율에는 `ice`와 `steam`을 구분하는 `solid`, `gas` 같은 의미 관계가 나타날 수 있으며, 이러한 구조가 벡터 차이에 반영된다.

## 3. 실행 환경

UV 프로젝트에 필요한 패키지가 없다면 노트북을 종료한 터미널에서 다음을 실행한다.

```bash
uv add gensim torch pandas matplotlib scikit-learn
```

설치 후 이 프로젝트의 가상환경을 사용하는 커널로 다시 시작한다.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from gensim.models import KeyedVectors
from gensim.utils import simple_preprocess
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

## 4. 작은 말뭉치 준비

아래 데이터는 GloVe의 동작을 빠르게 확인하기 위한 인공 말뭉치다. 실제 의미 품질을 얻으려면 수백만~수십억 개 수준의 토큰과 업무 도메인에 맞는 전처리가 필요하다.

`simple_preprocess()`는 영어 텍스트를 소문자로 만들고 간단히 토큰화한다. 한국어에서는 형태소 분석기 등을 이용해 별도로 토큰화한 결과를 사용하는 것이 일반적이다.

In [ ]:
raw_sentences = [
    "king queen royal palace",
    "king prince royal palace",
    "queen princess royal palace",
    "man king male royal",
    "woman queen female royal",
    "man boy male family",
    "woman girl female family",
    "cat kitten pet animal",
    "dog puppy pet animal",
    "cat dog pet home",
    "kitten puppy young pet",
    "boy girl young family",
]

# 반복은 작은 예제의 공기 빈도를 키워 최적화를 안정시키기 위한 장치다.
tokenized_corpus = [
    simple_preprocess(sentence)
    for sentence in raw_sentences
] * 100

print("문장 수:", len(tokenized_corpus))
print("첫 문장:", tokenized_corpus[0])

## 5. 단어-단어 공기행렬 만들기

한 문장에서 중심 단어의 좌우 `window_size` 범위 안에 있는 단어를 문맥 단어로 센다. 아래 구현은 가까운 문맥에 더 큰 값을 주기 위해 거리의 역수 $1/d$를 더한다.

예를 들어 `king queen royal palace`에서 `king`을 중심으로 window가 2라면 `queen`에는 1, 두 칸 떨어진 `royal`에는 0.5가 더해진다.

In [ ]:
def build_cooccurrence_matrix(tokenized_sentences, window_size=2):
    token_counts = Counter(
        token
        for sentence in tokenized_sentences
        for token in sentence
    )

    index_to_word = sorted(token_counts)
    word_to_index = {
        word: index
        for index, word in enumerate(index_to_word)
    }

    matrix = np.zeros(
        (len(index_to_word), len(index_to_word)),
        dtype=np.float32,
    )

    for sentence in tokenized_sentences:
        for center_position, center_word in enumerate(sentence):
            left = max(0, center_position - window_size)
            right = min(
                len(sentence),
                center_position + window_size + 1,
            )

            for context_position in range(left, right):
                if center_position == context_position:
                    continue

                context_word = sentence[context_position]
                distance = abs(center_position - context_position)

                center_id = word_to_index[center_word]
                context_id = word_to_index[context_word]
                matrix[center_id, context_id] += 1.0 / distance

    return matrix, word_to_index, index_to_word


cooccurrence, word_to_index, index_to_word = build_cooccurrence_matrix(
    tokenized_corpus,
    window_size=2,
)

print("Vocabulary 크기:", len(index_to_word))
print("공기행렬 크기:", cooccurrence.shape)

In [ ]:
cooccurrence_frame = pd.DataFrame(
    cooccurrence,
    index=index_to_word,
    columns=index_to_word,
)

selected_words = [
    "king", "queen", "man", "woman",
    "cat", "dog", "royal", "pet",
]

cooccurrence_frame.loc[
    selected_words,
    selected_words,
]

행렬의 한 행은 중심 단어가 어떤 문맥 단어와 함께 나타났는지를 보여 준다. 비슷한 문맥에서 사용되는 단어는 행의 패턴도 비슷해진다. 하지만 이 행렬 자체는 vocabulary가 커질수록 $V \times V$ 크기가 되어 매우 커진다. 실제 구현에서는 0이 아닌 원소만 저장하는 희소 구조와 병렬화된 학습 코드를 사용한다.

## 6. PyTorch로 미니 GloVe 학습

다음 구현은 학습 원리를 확인하기 위한 교육용 코드다. 중심 임베딩, 문맥 임베딩, 두 종류의 편향을 학습하며 공기 빈도가 0보다 큰 단어 쌍만 목적함수에 넣는다.

실제 대규모 학습에는 Stanford의 공식 GloVe 구현처럼 희소 행렬과 최적화된 코드를 사용하는 것이 적합하다.

In [ ]:
class MiniGloVe(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.center_embeddings = torch.nn.Embedding(
            vocab_size,
            embedding_dim,
        )
        self.context_embeddings = torch.nn.Embedding(
            vocab_size,
            embedding_dim,
        )
        self.center_biases = torch.nn.Embedding(vocab_size, 1)
        self.context_biases = torch.nn.Embedding(vocab_size, 1)

        torch.nn.init.normal_(
            self.center_embeddings.weight,
            std=0.1,
        )
        torch.nn.init.normal_(
            self.context_embeddings.weight,
            std=0.1,
        )
        torch.nn.init.zeros_(self.center_biases.weight)
        torch.nn.init.zeros_(self.context_biases.weight)

    def forward(self, center_ids, context_ids):
        center_vectors = self.center_embeddings(center_ids)
        context_vectors = self.context_embeddings(context_ids)

        dot_products = (
            center_vectors * context_vectors
        ).sum(dim=1)

        center_bias = self.center_biases(center_ids).squeeze(1)
        context_bias = self.context_biases(context_ids).squeeze(1)

        return dot_products + center_bias + context_bias


def glove_weight(values, x_max=100.0, alpha=0.75):
    return torch.clamp(
        (values / x_max) ** alpha,
        max=1.0,
    )

In [ ]:
# 공기 빈도가 0이 아닌 위치만 학습 데이터로 사용한다.
center_rows, context_columns = np.nonzero(cooccurrence)

center_ids = torch.tensor(center_rows, dtype=torch.long)
context_ids = torch.tensor(context_columns, dtype=torch.long)
cooccurrence_values = torch.tensor(
    cooccurrence[center_rows, context_columns],
    dtype=torch.float32,
)

mini_glove = MiniGloVe(
    vocab_size=len(index_to_word),
    embedding_dim=20,
)

optimizer = torch.optim.Adam(
    mini_glove.parameters(),
    lr=0.05,
)

loss_history = []

for epoch in range(501):
    optimizer.zero_grad()

    predictions = mini_glove(
        center_ids,
        context_ids,
    )

    weights = glove_weight(cooccurrence_values)
    targets = torch.log(cooccurrence_values)

    loss = (
        weights * (predictions - targets) ** 2
    ).mean()

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 100 == 0:
        print(f"epoch={epoch:>3}, loss={loss.item():.6f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Weighted squared error")
plt.title("Mini GloVe training loss")
plt.grid(alpha=0.3)
plt.show()

## 7. 학습된 단어 벡터 사용

중심 벡터와 문맥 벡터의 합을 최종 임베딩으로 사용한다. 두 벡터의 방향이 얼마나 비슷한지는 코사인 유사도로 측정한다.

$$
\operatorname{cosine}(a,b)
=
\frac{a \cdot b}
{\lVert a \rVert_2\lVert b \rVert_2}
$$

1에 가까울수록 방향이 유사하지만, 높은 값이 반드시 동의어라는 뜻은 아니다. 반의어나 같은 주제에서 자주 등장하는 단어도 가까울 수 있다.

In [ ]:
with torch.no_grad():
    toy_vectors = (
        mini_glove.center_embeddings.weight
        + mini_glove.context_embeddings.weight
    ).cpu().numpy()


def most_similar_from_array(
    word,
    vectors,
    word_to_index,
    index_to_word,
    topn=5,
):
    if word not in word_to_index:
        raise KeyError(f"Vocabulary에 없는 단어입니다: {word}")

    query_vector = vectors[word_to_index[word]]
    vector_norms = np.linalg.norm(
        vectors,
        axis=1,
    )
    query_norm = np.linalg.norm(query_vector)

    scores = (
        vectors @ query_vector
    ) / (vector_norms * query_norm + 1e-9)

    sorted_indices = np.argsort(-scores)

    results = []
    for index in sorted_indices:
        candidate = index_to_word[index]

        if candidate == word:
            continue

        results.append((candidate, float(scores[index])))

        if len(results) == topn:
            break

    return results


for query in ["king", "queen", "cat", "dog", "man", "woman"]:
    print(query, most_similar_from_array(
        query,
        toy_vectors,
        word_to_index,
        index_to_word,
    ))

작은 인공 데이터에서는 결과가 말뭉치 문장을 그대로 반영하며 일반적인 영어 의미를 학습했다고 볼 수 없다. 예를 들어 `king`과 `queen`이 가깝게 나오는 이유는 이 예제에서 두 단어가 `royal`, `palace` 같은 문맥을 공유하기 때문이다. 데이터와 초기값이 바뀌면 순위도 바뀔 수 있다.

## 8. Gensim으로 사전학습 GloVe 사용

실무에서는 GloVe를 처음부터 직접 학습하기보다 공개 사전학습 벡터를 기준선으로 사용하는 경우가 많다. Gensim Downloader는 Stanford GloVe 벡터를 `KeyedVectors` 형식으로 내려받아 유사도 조회에 사용할 수 있게 한다.

`glove-wiki-gigaword-50`은 Wikipedia 2014와 Gigaword 5를 기반으로 학습된 400,000개 vocabulary, 50차원 영문 벡터다. 첫 실행에는 약 66MB 수준의 파일을 다운로드하며, 메모리에 적재할 때는 더 많은 공간이 필요하다.

아래 셀은 의도하지 않은 다운로드를 막기 위해 기본값이 `False`다. 인터넷 연결과 디스크 여유를 확인한 후 `True`로 바꾼다.

공식 자료: [Gensim Downloader](https://radimrehurek.com/gensim/downloader.html)

In [ ]:
import gensim.downloader as api

DOWNLOAD_PRETRAINED = False
PRETRAINED_MODEL_NAME = "glove-wiki-gigaword-50"

if DOWNLOAD_PRETRAINED:
    model_info = api.info(PRETRAINED_MODEL_NAME)
    print("모델 설명:", model_info["description"])

    glove_vectors = api.load(PRETRAINED_MODEL_NAME)
    print("Vocabulary 크기:", len(glove_vectors))
    print("벡터 차원:", glove_vectors.vector_size)
else:
    glove_vectors = None
    print(
        "다운로드를 실행하려면 DOWNLOAD_PRETRAINED=True로 변경하세요."
    )

### 유사어와 단어 유추

`most_similar()`는 기본적으로 코사인 유사도가 높은 단어를 반환한다. 단어 유추는 벡터의 선형 구조를 이용한다.

$$
v_{\text{king}} - v_{\text{man}} + v_{\text{woman}}
\approx v_{\text{queen}}
$$

유명한 예제이지만 모든 관계가 안정적인 선형 연산으로 표현되는 것은 아니다. 결과는 학습 말뭉치의 시대, 문화, 빈도와 편향을 반영한다.

In [ ]:
if glove_vectors is not None:
    print("computer와 유사한 단어")
    for word, score in glove_vectors.most_similar(
        "computer",
        topn=10,
    ):
        print(f"{word:<15} {score:.4f}")

    print("\nking - man + woman")
    analogy = glove_vectors.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=5,
    )
    print(analogy)

    print(
        "\ncat-dog 유사도:",
        glove_vectors.similarity("cat", "dog"),
    )
else:
    print("먼저 사전학습 벡터 다운로드 셀을 실행하세요.")

### OOV(Out Of Vocabulary) 안전하게 처리하기

GloVe는 학습 vocabulary에 없는 단어의 벡터를 만들 수 없다. 운영 코드에서는 조회 전에 포함 여부를 확인해야 한다. 대소문자와 전처리가 달라져도 OOV가 발생할 수 있으며, 위 모델은 소문자 기반이다.

In [ ]:
def safe_get_vector(word, keyed_vectors):
    normalized_word = word.lower()

    if normalized_word not in keyed_vectors:
        return None

    return keyed_vectors[normalized_word]


if glove_vectors is not None:
    for word in ["computer", "Computer", "not_a_real_word_123"]:
        vector = safe_get_vector(word, glove_vectors)
        status = "벡터 있음" if vector is not None else "OOV"
        print(f"{word:<24} {status}")

## 9. 단어 벡터 시각화

50차원 벡터를 PCA로 2차원에 투영한다. 2차원 그림은 고차원 거리의 일부만 보존하므로 가까워 보인다는 사실을 정량 평가로 오해하면 안 된다.

In [ ]:
visualization_words = [
    "king", "queen", "man", "woman",
    "paris", "france", "berlin", "germany",
    "cat", "dog", "apple", "orange",
]

if glove_vectors is not None:
    valid_words = [
        word
        for word in visualization_words
        if word in glove_vectors
    ]
    matrix = np.vstack([
        glove_vectors[word]
        for word in valid_words
    ])

    coordinates = PCA(
        n_components=2,
    ).fit_transform(matrix)

    plt.figure(figsize=(10, 7))
    plt.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
    )

    for word, (x, y) in zip(valid_words, coordinates):
        plt.annotate(
            word,
            (x, y),
            xytext=(5, 5),
            textcoords="offset points",
        )

    plt.title("PCA projection of GloVe vectors")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("먼저 사전학습 벡터 다운로드 셀을 실행하세요.")

## 10. 문장을 고정 길이 벡터로 변환

GloVe는 단어 임베딩이므로 문서 분류기에는 문서 단위 벡터가 필요하다. 가장 간단한 기준선은 문서에 포함된 단어 벡터의 평균이다.

$$
v_{\text{document}}
=
\frac{1}{N}\sum_{i=1}^{N}v_{w_i}
$$

단순 평균은 단어 순서와 중요도 차이를 잃지만 빠르고 해석하기 쉬워 분류·군집화 기준선으로 유용하다. 모든 단어가 OOV이면 영벡터를 반환하되, 운영에서는 이런 문서 비율을 별도 모니터링해야 한다.

In [ ]:
def mean_glove_embedding(text, keyed_vectors):
    tokens = simple_preprocess(text)
    valid_vectors = [
        keyed_vectors[token]
        for token in tokens
        if token in keyed_vectors
    ]

    if not valid_vectors:
        return np.zeros(
            keyed_vectors.vector_size,
            dtype=np.float32,
        )

    return np.mean(valid_vectors, axis=0)


if glove_vectors is not None:
    sample_documents = [
        "The delivery was fast and the product was excellent",
        "The package arrived broken and late",
        "The football team won the championship",
    ]

    document_matrix = np.vstack([
        mean_glove_embedding(document, glove_vectors)
        for document in sample_documents
    ])

    print("문서 행렬 크기:", document_matrix.shape)
else:
    print("먼저 사전학습 벡터 다운로드 셀을 실행하세요.")

실무에서는 평균 벡터를 Logistic Regression, LightGBM, 군집화 알고리즘 등에 입력할 수 있다. 단순 평균보다 개선하려면 TF-IDF 가중 평균, 도메인 말뭉치로 학습한 임베딩, FastText 또는 문맥형 Transformer 임베딩과 비교한다. 임베딩을 검증·테스트 문서까지 사용해 다시 학습하면 데이터 분포 정보가 누출될 수 있으므로 평가 설계도 함께 고정해야 한다.

## 11. 로컬 GloVe 파일 불러오기

Stanford에서 받은 원본 GloVe 텍스트 파일은 첫 줄에 vocabulary 크기와 차원 정보가 없는 형식이다. Gensim 4.x에서는 `no_header=True`로 불러올 수 있다.

각 줄은 다음과 같이 단어 뒤에 실수 벡터가 이어지는 구조다.

```text
the 0.418 0.24968 -0.41242 ...
of 0.70853 0.57088 -0.4716 ...
```

In [ ]:
# 실제 파일을 내려받은 후 경로와 실행 여부를 변경한다.
LOAD_LOCAL_GLOVE = False
local_glove_path = Path("data/glove.6B.50d.txt")

if LOAD_LOCAL_GLOVE:
    local_vectors = KeyedVectors.load_word2vec_format(
        local_glove_path,
        binary=False,
        no_header=True,
    )
    print(len(local_vectors), local_vectors.vector_size)
else:
    print("LOAD_LOCAL_GLOVE=True로 변경하면 로컬 파일을 읽습니다.")

## 12. 저장과 다시 불러오기

다운로드된 벡터는 Gensim 캐시에 저장되지만, 서비스 배포에서는 검증한 벡터를 애플리케이션 산출물로 고정하는 것이 안전하다. `KeyedVectors` 형식은 조회 전용이며 GloVe를 추가 학습하기 위한 공기행렬이나 옵티마이저 상태는 포함하지 않는다.

큰 파일은 `mmap="r"`로 읽으면 여러 프로세스가 읽기 전용 메모리 매핑을 공유할 수 있다.

In [ ]:
SAVE_PRETRAINED = False
artifact_path = Path("artifacts/glove_wiki_gigaword_50.kv")

if SAVE_PRETRAINED and glove_vectors is not None:
    artifact_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    glove_vectors.save(str(artifact_path))

    reloaded_vectors = KeyedVectors.load(
        str(artifact_path),
        mmap="r",
    )

    print("다시 불러온 벡터:", reloaded_vectors["computer"][:5])
else:
    print(
        "저장하려면 벡터를 내려받고 SAVE_PRETRAINED=True로 변경하세요."
    )

## 13. 실무 체크리스트

### 데이터와 전처리

- 사전학습 말뭉치의 언어·시대·도메인이 실제 데이터와 맞는지 확인한다.
- 학습과 추론에서 대소문자, 숫자, 구두점 처리 규칙을 동일하게 유지한다.
- 실제 입력의 OOV 비율과 빈 문서 벡터 비율을 측정한다.
- 한국어에서는 공백 분리 대신 업무 목적에 맞는 형태소 분석을 검토한다.

### 모델 선택

- 빠르고 가벼운 기준선에는 TF-IDF와 GloVe 평균 벡터가 유용하다.
- 오타와 신규 단어가 많으면 문자 n-gram을 사용하는 FastText를 비교한다.
- 문맥에 따른 다의어와 문장 순서가 중요하면 Transformer 임베딩을 비교한다.

### 평가와 배포

- 유사 단어의 정성 평가만 하지 말고 분류 F1, 검색 Recall@K 등 최종 업무 지표를 측정한다.
- 사전학습 벡터의 라이선스를 확인한다.
- 벡터 파일, 전처리 코드, 분류 모델을 같은 버전으로 관리한다.
- 임베딩을 바꾸면 좌표계가 달라지므로 기존 분류기도 다시 학습하고 검증한다.
- 운영 시작 시 인터넷에서 다운로드하지 말고 검증된 파일을 배포 이미지나 모델 저장소에 고정한다.

GloVe는 구현이 단순하고 추론이 빠르며 의미 있는 기준선을 만들기 좋다. 그러나 최신 방법이라는 이유만으로 Transformer를 선택하거나, 반대로 가볍다는 이유만으로 GloVe를 고집하기보다 정확도·지연시간·메모리·운영 복잡도를 함께 비교해야 한다.